In [1]:
import torch

x = torch.rand(5000, 5000).cuda()
y = torch.mm(x, x)

print(y.device)

cuda:0


In [2]:
!pip uninstall sentence-transformers -y
!pip install sentence-transformers==2.7.0

Found existing installation: sentence-transformers 2.7.0
Uninstalling sentence-transformers-2.7.0:
  Successfully uninstalled sentence-transformers-2.7.0
  Using cached sentence_transformers-2.7.0-py3-none-any.whl.metadata (11 kB)
Using cached sentence_transformers-2.7.0-py3-none-any.whl (171 kB)



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install transformers==4.48.3

  Using cached tokenizers-0.21.4-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 9.7/9.7 MB 50.0 MB/s eta 0:00:00
Using cached tokenizers-0.21.4-cp39-abi3-win_amd64.whl (2.5 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import gc
import os
import sys
import json
import re

# =========================
# CUDA / 메모리 초기화
# =========================
import torch

gc.collect()
torch.cuda.empty_cache()

# =========================
# 기본 라이브러리
# =========================
import pandas as pd
import torch.nn.functional as F

# =========================
# HuggingFace 계열 먼저 로드
# =========================
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# =========================
# sklearn
# =========================
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

# =========================
# sentence-transformers는 맨 마지막
# =========================
from sentence_transformers import (
    SentenceTransformer,
    util
)

In [3]:
# =========================================================
# GPU 확인 및 작업 경로 체크
# =========================================================

print("=" * 80)
print("환경 검증 및 GPU 확인")
print("=" * 80)

print("현재 파이썬 작업 경로(PWD):", os.getcwd())
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
print("=" * 80)

환경 검증 및 GPU 확인
현재 파이썬 작업 경로(PWD): c:\Users\AI_SVR05\Desktop\NLP
CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 3070


In [4]:
# =========================================================
# device 설정
# SBERT는 CPU
# KoELECTRA는 GPU
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# =========================================================
# 메모리 정리
# =========================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =========================================================
# 모델 설정
# =========================================================

MODEL_NAME = "monologg/koelectra-small-v3-discriminator"

# =========================================================
# SBERT 모델
# CPU 사용 (중요)
# =========================================================

similarity_model = SentenceTransformer(
    "snunlp/KR-SBERT-V40K-klueNLI-augSTS",
    device="cpu"
)

# =========================================================
# 중요 패턴
# =========================================================

important_patterns = [
    "중요", "핵심", "반드시", "시험", "출제", "필수", 
    "정의", "의미", "란", "특징", "역할", "구조", 
    "원리", "과정", "개념", "주의"
]

# =========================================================
# hard negative 패턴
# =========================================================

hard_negative_patterns = [
    "최근", "설명하였다", "발표하였다", "소개하였다", 
    "진행하였다", "언급하였다", "보고하였다"
]

# =========================================================
# 문장 분리
# =========================================================

def split_sentences(text):
    if text is None:
        return []

    text = str(text)
    sentences = re.split(r'(?<=[.!?다])\s+', text)

    return [
        s.strip()
        for s in sentences
        if len(s.strip()) > 5
    ]

# =========================================================
# 핵심 개념 추출
# =========================================================

def extract_keywords(sentence):
    keywords = []
    patterns = [
        r"[가-힣A-Za-z0-9]+ 모델",
        r"[가-힣A-Za-z0-9]+ 알고리즘",
        r"[가-힣A-Za-z0-9]+ 함수",
        r"[가-힣A-Za-z0-9]+ 구조",
        r"[가-힣A-Za-z0-9]+ 기법",
        r"[가-힣A-Za-z0-9]+ 시스템",
        r"[가-힣A-Za-z0-9]+ 네트워크",
        r"[가-힣A-Za-z0-9]+ 학습",
        r"[가-힣A-Za-z0-9]+ 분석",
        r"[A-Z]{2,}"
    ]

    for pattern in patterns:
        found = re.findall(pattern, sentence)
        for word in found:
            if word not in keywords:
                keywords.append(word)

    return keywords

# =========================================================
# 데이터셋 생성
# =========================================================

def build_dataset(jsonl_path, max_data=300):
    # 파일이 진짜 존재하는지 체크하는 안전장치
    if not os.path.exists(jsonl_path):
        print(f"❌ 에러: [{jsonl_path}] 파일을 찾을 수 없습니다. 경로를 다시 확인하세요.")
        sys.exit(1)

    dataset = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx >= max_data:
                break

            item = json.loads(line)
            input_text = item["input"]
            summary_text = item["output"]

            sentences = split_sentences(input_text)
            if len(sentences) == 0:
                continue

            # -------------------------------------------------
            # SBERT 임베딩
            # -------------------------------------------------
            sentence_embeddings = similarity_model.encode(
                sentences,
                convert_to_tensor=True,
                show_progress_bar=False
            )

            summary_embedding = similarity_model.encode(
                summary_text,
                convert_to_tensor=True,
                show_progress_bar=False
            )

            # -------------------------------------------------
            # 문장별 라벨 생성
            # -------------------------------------------------
            for s_idx, sentence in enumerate(sentences):
                similarity = util.cos_sim(
                    sentence_embeddings[s_idx],
                    summary_embedding
                ).item()

                rule_based = any(p in sentence for p in important_patterns)
                hard_negative = any(p in sentence for p in hard_negative_patterns)

                label = 1 if (
                    (similarity >= 0.35 or rule_based)
                    and not hard_negative
                ) else 0

                dataset.append({
                    "text": sentence,
                    "label": label
                })

            # -------------------------------------------------
            # 메모리 정리
            # -------------------------------------------------
            del sentence_embeddings
            del summary_embedding
            gc.collect()

    return pd.DataFrame(dataset)

# =========================================================
# 데이터셋 생성 시작 (★ 경로 수정: koelectra_project/ 추가)
# =========================================================

print("\ntrain 데이터 생성 중...")
train_df = build_dataset("koelectra_project/train.jsonl", max_data=300)

print("valid 데이터 생성 중...")
valid_df = build_dataset("koelectra_project/valid.jsonl", max_data=100)

print("test 데이터 생성 중...")
test_df = build_dataset("koelectra_project/test.jsonl", max_data=100)

print("\n데이터 생성 완료")
print(f"train 개수: {len(train_df)}")
print(f"valid 개수: {len(valid_df)}")
print(f"test 개수 : {len(test_df)}")

# =========================================================
# 클래스 균형 맞추기
# =========================================================

positive_df = train_df[train_df["label"] == 1]
negative_df = train_df[train_df["label"] == 0]

negative_df = negative_df.sample(
    n=min(len(negative_df), len(positive_df) * 2),
    random_state=42
)

train_df = pd.concat([positive_df, negative_df])
train_df = train_df.sample(frac=1, random_state=42)

print("\n라벨 분포")
print(train_df["label"].value_counts())

# =========================================================
# Dataset 변환
# =========================================================

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

# =========================================================
# 토크나이저
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================================
# 토큰화
# =========================================================

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# =========================================================
# torch format
# =========================================================

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
valid_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =========================================================
# 모델 로드
# =========================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
model.to(device)

# =========================================================
# 평가 함수
# =========================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1
    }

# =========================================================
# 학습 옵션
# =========================================================

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=20,
    fp16=False,
    load_best_model_at_end=True
)

# =========================================================
# Trainer
# =========================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# =========================================================
# 학습 시작
# =========================================================

print("\n")
print("=" * 80)
print("KoELECTRA 학습 시작")
print("=" * 80)

trainer.train()

# =========================================================
# 최종 평가
# =========================================================

results = trainer.evaluate(test_dataset)

print("\n")
print("=" * 80)
print("최종 성능")
print("=" * 80)

print(f"Accuracy : {results['eval_accuracy']:.4f}")
print(f"F1 Score : {results['eval_f1']:.4f}")
print(f"Loss      : {results['eval_loss']:.4f}")

# =========================================================
# 모델 저장
# =========================================================

SAVE_PATH = "./koelectra_keysentence_model"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("\n모델 저장 완료")

# =========================================================
# 예측 함수
# =========================================================

def predict_sentence(sentence):
    model.eval()
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][1].item()

    return pred, confidence

# =========================================================
# 긴 문장 테스트
# =========================================================

print("\n")
print("=" * 80)
print("긴 문장 기반 모델 성능 테스트")
print("=" * 80)

sample_text = """
딥러닝이란 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로,
대량의 데이터를 활용해 이미지 분류, 자연어 처리, 음성 인식 등의 다양한 문제를 해결할 수 있다.

CNN 모델은 이미지의 공간적 특징을 효과적으로 추출할 수 있기 때문에
컴퓨터 비전 분야에서 매우 중요한 역할을 수행하며,
시험에서도 자주 출제되는 핵심 개념 중 하나이다.

Transformer 구조는 Self-Attention 메커니즘을 기반으로 문장 내 단어 간의 관계를 계산하며,
기존 RNN 기반 모델보다 병렬 처리가 가능하다는 장점이 있다.

오늘은 날씨가 좋아서 친구들과 함께 카페에 가서 커피를 마셨고,
저녁에는 영화를 보며 시간을 보냈다.

GPU는 대규모 행렬 연산을 병렬로 처리할 수 있기 때문에
딥러닝 모델 학습 속도를 크게 향상시킬 수 있으며,
CUDA 환경 설정은 실습에서 매우 중요하게 다뤄진다.
"""

sentences = split_sentences(sample_text)

for idx, sentence in enumerate(sentences):
    pred, confidence = predict_sentence(sentence)

    print(f"\n[{idx+1}] 문장")
    print(sentence)
    print(f"\n핵심 문장 확률: {confidence:.4f}")

    if confidence >= 0.40:
        print("\n예측 결과:")
        print("핵심 문장")
        keywords = extract_keywords(sentence)
        print("\n핵심 개념:")
        print(keywords)
    else:
        print("\n예측 결과:")
        print("일반 문장")

    print("-" * 80)

loading configuration file config.json from cache at C:\Users\AI_SVR05\.cache\huggingface\hub\models--snunlp--KR-SBERT-V40K-klueNLI-augSTS\snapshots\92c6c2c7032f680bff0f9f0c63fadd3f97e635b2\config.json
Model config BertConfig {
  "_name_or_path": "snunlp/KR-SBERT-V40K-klueNLI-augSTS",
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.48.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 40000
}

loading weights file pytorch_model.bin from cache at C:\Users\AI_SVR05\.cache\huggingface\hub\mod


train 데이터 생성 중...


Attempting to create safetensors variant
Safetensors PR exists


valid 데이터 생성 중...
test 데이터 생성 중...

데이터 생성 완료
train 개수: 3846
valid 개수: 1234
test 개수 : 1222

라벨 분포
label
1    2792
0    1054
Name: count, dtype: int64


loading configuration file config.json from cache at C:\Users\AI_SVR05\.cache\huggingface\hub\models--monologg--koelectra-small-v3-discriminator\snapshots\7488f8db0f208beff4a1f3f9bb3ed04650a89ed7\config.json
Model config ElectraConfig {
  "_name_or_path": "monologg/koelectra-small-v3-discriminator",
  "architectures": [
    "ElectraForPreTraining"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "embedding_size": 128,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 256,
  "initializer_range": 0.02,
  "intermediate_size": 1024,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "electra",
  "num_attention_heads": 4,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "summary_activation": "gelu",
  "summary_last_dropout": 0.1,
  "summary_type": "first",
  "summary_use_proj": true,
  "transformers_version": "4.48.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size

...

Attempting to create safetensors variant


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.541300,0.518707,0.774716,0.858883


Safetensors PR exists
Saving model checkpoint to ./results\checkpoint-1923
Configuration saved in ./results\checkpoint-1923\config.json
Model weights saved in ./results\checkpoint-1923\model.safetensors
The following columns in the evaluation set don't have a corresponding argument in `ElectraForSequenceClassification.forward` and have been ignored: text. If text are not expected by `ElectraForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1234
  Batch size = 2
Saving model checkpoint to ./results\checkpoint-1923
Configuration saved in ./results\checkpoint-1923\config.json
Model weights saved in ./results\checkpoint-1923\model.safetensors


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./results\checkpoint-1923 (score: 0.5187065601348877).
The following columns in the evaluation set don't have a corresponding argument in `ElectraForSequenceClassification

Saving model checkpoint to ./koelectra_keysentence_model
Configuration saved in ./koelectra_keysentence_model\config.json
Model weights saved in ./koelectra_keysentence_model\model.safetensors
tokenizer config file saved in ./koelectra_keysentence_model\tokenizer_config.json
Special tokens file saved in ./koelectra_keysentence_model\special_tokens_map.json




최종 성능
Accuracy : 0.8118
F1 Score : 0.8814
Loss      : 0.4681

모델 저장 완료


긴 문장 기반 모델 성능 테스트

[1] 문장
딥러닝이란 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로,
대량의 데이터를 활용해 이미지 분류, 자연어 처리, 음성 인식 등의 다양한 문제를 해결할 수 있다.

핵심 문장 확률: 0.8918

예측 결과:
핵심 문장

핵심 개념:
['신경망 구조', '머신러닝 기법', '데이터를 학습']
--------------------------------------------------------------------------------

[2] 문장
CNN 모델은 이미지의 공간적 특징을 효과적으로 추출할 수 있기 때문에
컴퓨터 비전 분야에서 매우 중요한 역할을 수행하며,
시험에서도 자주 출제되는 핵심 개념 중 하나이다.

핵심 문장 확률: 0.8912

예측 결과:
핵심 문장

핵심 개념:
['CNN 모델', 'CNN']
--------------------------------------------------------------------------------

[3] 문장
Transformer 구조는 Self-Attention 메커니즘을 기반으로 문장 내 단어 간의 관계를 계산하며,
기존 RNN 기반 모델보다

핵심 문장 확률: 0.8909

예측 결과:
핵심 문장

핵심 개념:
['기반 모델', 'Transformer 구조', 'RNN']
--------------------------------------------------------------------------------

[4] 문장
병렬 처리가 가능하다는 장점이 있다.

핵심 문장 확률: 0.3021

예측 결과:
일반 문장
--------------------------------------------------------------------------------

[5] 문장
오늘은 날씨가 좋

In [5]:
# =========================================================
# 학습 완료 후 성능 테스트 코드
# 긴 강의자료 스타일 문장 테스트
# =========================================================

print("\n")
print("=" * 100)
print("KoELECTRA 핵심 문장 판별 테스트")
print("=" * 100)

sample_text = """

딥러닝은 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로,
대량의 데이터를 활용해 이미지 분류, 자연어 처리, 음성 인식 등의 다양한 문제를 해결할 수 있다.
특히 최근에는 생성형 인공지능 기술이 발전하면서 딥러닝의 중요성이 더욱 강조되고 있다.

CNN 모델은 이미지의 공간적 특징을 효과적으로 추출할 수 있기 때문에
컴퓨터 비전 분야에서 매우 중요한 역할을 수행한다.
합성곱 연산을 활용하여 특징을 추출하며,
Pooling Layer를 통해 차원을 축소하고 연산량을 감소시킨다.
시험에서는 CNN의 구조와 특징이 자주 출제되는 핵심 개념 중 하나이다.

Transformer 구조는 Self-Attention 메커니즘을 기반으로 문장 내 단어 간의 관계를 계산하며,
기존 RNN 기반 모델보다 병렬 처리가 가능하다는 장점이 있다.
특히 BERT와 GPT 같은 최신 자연어 처리 모델의 기반 구조로 사용되며,
최근 자연어 처리 분야에서 가장 중요한 핵심 개념으로 평가받고 있다.

데이터 전처리 과정에서는 결측치 제거, 정규화, 이상치 탐지 등의 작업이 수행된다.
이 과정은 모델의 성능을 향상시키기 위해 반드시 필요하며,
데이터 품질이 낮을 경우 모델의 정확도가 크게 감소할 수 있다.

GPU는 대규모 행렬 연산을 병렬로 처리할 수 있기 때문에
딥러닝 모델 학습 속도를 크게 향상시킬 수 있다.
특히 CUDA 환경 설정은 딥러닝 실습에서 매우 중요하게 다뤄지며,
GPU 메모리 관리 또한 모델 성능 최적화에 중요한 역할을 한다.

시험 문제에서는 Softmax 함수의 역할과 Cross Entropy Loss의 계산 과정을
비교하여 설명하는 문제가 자주 출제된다.
또한 활성화 함수의 종류와 특징을 비교하는 문제 역시 핵심 유형으로 자주 등장한다.

오늘은 날씨가 좋아서 친구들과 함께 카페에 가서 커피를 마셨고,
저녁에는 영화를 보며 시간을 보냈다.
주말에는 여행을 가기로 계획했으며,
맛집을 검색하면서 시간을 보내기도 했다.

최근 학교 축제에서는 다양한 공연과 이벤트가 진행되었으며,
학생들이 많이 참여하여 분위기가 매우 활발했다.
동아리 부스에서는 여러 가지 게임과 체험 활동이 운영되었다.

KoELECTRA는 ELECTRA 구조를 기반으로 학습된 한국어 언어 모델이며,
문장 분류와 감정 분석, 자연어 이해 분야에서 높은 성능을 보이는 대표적인 모델이다.
특히 적은 데이터셋에서도 우수한 성능을 보인다는 특징이 있다.

"""

# =========================================================
# 문장 분리
# =========================================================

sentences = split_sentences(sample_text)

# =========================================================
# 핵심 문장 판별
# =========================================================

for idx, sentence in enumerate(sentences):

    pred, confidence = predict_sentence(sentence)

    print("\n")
    print("=" * 100)
    print(f"[문장 {idx+1}]")
    print("=" * 100)

    print(sentence)

    print(f"\n핵심 문장 확률: {confidence:.4f}")

    # -----------------------------------------------------
    # 핵심 문장
    # -----------------------------------------------------

    if confidence >= 0.40:

        print("\n예측 결과:")
        print("핵심 문장")

        keywords = extract_keywords(sentence)

        print("\n핵심 개념:")

        if len(keywords) > 0:

            for keyword in keywords:

                print(f"- {keyword}")

        else:

            print("- 추출된 핵심 개념 없음")

    # -----------------------------------------------------
    # 일반 문장
    # -----------------------------------------------------

    else:

        print("\n예측 결과:")
        print("일반 문장")

    print("\n")



KoELECTRA 핵심 문장 판별 테스트


[문장 1]
딥러닝은 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로,
대량의 데이터를 활용해 이미지 분류, 자연어 처리, 음성 인식 등의 다양한 문제를 해결할 수 있다.

핵심 문장 확률: 0.8917

예측 결과:
핵심 문장

핵심 개념:
- 신경망 구조
- 머신러닝 기법
- 데이터를 학습




[문장 2]
특히 최근에는 생성형 인공지능 기술이 발전하면서 딥러닝의 중요성이 더욱 강조되고 있다.

핵심 문장 확률: 0.8913

예측 결과:
핵심 문장

핵심 개념:
- 추출된 핵심 개념 없음




[문장 3]
CNN 모델은 이미지의 공간적 특징을 효과적으로 추출할 수 있기 때문에
컴퓨터 비전 분야에서 매우 중요한 역할을 수행한다.

핵심 문장 확률: 0.8914

예측 결과:
핵심 문장

핵심 개념:
- CNN 모델
- CNN




[문장 4]
합성곱 연산을 활용하여 특징을 추출하며,
Pooling Layer를 통해 차원을 축소하고 연산량을 감소시킨다.

핵심 문장 확률: 0.8910

예측 결과:
핵심 문장

핵심 개념:
- 추출된 핵심 개념 없음




[문장 5]
시험에서는 CNN의 구조와 특징이 자주 출제되는 핵심 개념 중 하나이다.

핵심 문장 확률: 0.8916

예측 결과:
핵심 문장

핵심 개념:
- CNN의 구조
- CNN




[문장 6]
Transformer 구조는 Self-Attention 메커니즘을 기반으로 문장 내 단어 간의 관계를 계산하며,
기존 RNN 기반 모델보다

핵심 문장 확률: 0.8909

예측 결과:
핵심 문장

핵심 개념:
- 기반 모델
- Transformer 구조
- RNN




[문장 7]
병렬 처리가 가능하다는 장점이 있다.

핵심 문장 확률: 0.3021

예측 결과:
일반 문장




[문장 8]
특히 BERT와 GPT 같은 최신 자연어 처리 모델의 기반 구조로 사용되며,
최근 자연어 처리 분야에서 가장 중요한 핵심 개념으로 평가받고

In [ ]:
# =========================================================
# KoBART 생성형 요약 추가 코드
# KoELECTRA 코드는 수정하지 않음
# =========================================================

# 필요 라이브러리
# !pip install transformers sentencepiece -q

from transformers import (
    PreTrainedTokenizerFast,
    BartForConditionalGeneration
)

# =========================================================
# KoBART 모델 로드
# =========================================================

print("=" * 80)
print("KoBART 생성형 요약 모델 로드")
print("=" * 80)

kobart_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "digit82/kobart-summarization"
)

kobart_model = BartForConditionalGeneration.from_pretrained(
    "digit82/kobart-summarization"
).to(device)

print("KoBART 로드 완료")
print("=" * 80)

# =========================================================
# 핵심 문장 추출 함수
# (KoELECTRA 기존 함수 그대로 사용)
# =========================================================

def extract_core_sentences(text, threshold=0.8):

    sentences = split_sentences(text)

    core_sentences = []

    for sentence in sentences:

        pred, confidence = predict_sentence(sentence)

        # 중요 문장만 선택
        if pred == 1 and confidence >= threshold:

            core_sentences.append(sentence)

    return core_sentences

# =========================================================
# KoBART 생성형 요약 함수
# =========================================================

def generate_summary(
    text,
    threshold=0.8,
    max_input_length=700
):

    # -----------------------------------------------------
    # 1. 핵심 문장 추출
    # -----------------------------------------------------

    core_sentences = extract_core_sentences(
        text,
        threshold=threshold
    )

    # -----------------------------------------------------
    # 2. 핵심 문장 없을 경우
    # -----------------------------------------------------

    if len(core_sentences) == 0:

        print("핵심 문장이 없어 원문 일부 사용")

        core_text = text[:max_input_length]

    else:

        # -------------------------------------------------
        # 문장 정제
        # -------------------------------------------------

        filtered = []

        for sent in core_sentences:

            sent = sent.strip()

            # 너무 짧은 문장 제거
            if len(sent) < 15:
                continue

            # 중복 제거
            if sent not in filtered:
                filtered.append(sent)

        # -------------------------------------------------
        # 입력 길이 제한
        # -------------------------------------------------

        core_text = ""

        for sent in filtered:

            if len(core_text) + len(sent) > max_input_length:
                break

            core_text += sent + " "

    # -----------------------------------------------------
    # 3. 토큰화
    # -----------------------------------------------------

    inputs = kobart_tokenizer(
        core_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # -----------------------------------------------------
    # 4. 생성형 요약
    # -----------------------------------------------------

    summary_ids = kobart_model.generate(

        input_ids=input_ids,
        attention_mask=attention_mask,

        # 요약 길이
        max_length=180,
        min_length=60,

        # Beam Search
        num_beams=8,

        # 반복 방지
        repetition_penalty=1.8,
        no_repeat_ngram_size=3,

        # 자연스러운 길이
        length_penalty=1.2,

        # 다양성 증가
        do_sample=True,
        top_k=50,
        top_p=0.95,

        early_stopping=True
    )

    # -----------------------------------------------------
    # 5. 디코딩
    # -----------------------------------------------------

    summary = kobart_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return core_sentences, summary

# =========================================================
# 테스트용 긴 문서
# =========================================================

long_text = """

딥러닝은 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로,
대량의 데이터를 활용해 이미지 분류, 자연어 처리, 음성 인식 등의 다양한 문제를 해결할 수 있다.
특히 최근에는 생성형 인공지능 기술이 발전하면서 딥러닝의 중요성이 더욱 강조되고 있다.

CNN 모델은 이미지의 공간적 특징을 효과적으로 추출할 수 있기 때문에
컴퓨터 비전 분야에서 매우 중요한 역할을 수행한다.
합성곱 연산을 활용하여 특징을 추출하며,
Pooling Layer를 통해 차원을 축소하고 연산량을 감소시킨다.

Transformer 구조는 Self-Attention 메커니즘을 기반으로 문장 내 단어 간의 관계를 계산하며,
기존 RNN 기반 모델보다 병렬 처리가 가능하다는 장점이 있다.
특히 BERT와 GPT 같은 최신 자연어 처리 모델의 기반 구조로 사용된다.

데이터 전처리 과정에서는 결측치 제거, 정규화, 이상치 탐지 등의 작업이 수행된다.
이 과정은 모델의 성능을 향상시키기 위해 반드시 필요하다.

GPU는 대규모 행렬 연산을 병렬로 처리할 수 있기 때문에
딥러닝 모델 학습 속도를 크게 향상시킬 수 있다.
특히 CUDA 환경 설정은 딥러닝 실습에서 매우 중요하게 다뤄진다.

KoELECTRA는 ELECTRA 구조를 기반으로 학습된 한국어 언어 모델이며,
문장 분류와 감정 분석 분야에서 높은 성능을 보인다.

"""

# =========================================================
# 생성형 요약 실행
# =========================================================

core_sentences, summary = generate_summary(
    long_text,
    threshold=0.8
)

# =========================================================
# 결과 출력
# =========================================================

print("\n")
print("=" * 100)
print("추출된 핵심 문장")
print("=" * 100)

for idx, sentence in enumerate(core_sentences):

    print(f"\n[{idx+1}] {sentence}")

print("\n")
print("=" * 100)
print("최종 생성형 요약")
print("=" * 100)

print(summary)

print("\n")
print("=" * 100)
print("요약 완료")
print("=" * 100)

print("\n")
print("=" * 100)
print("최종 생성형 요약문")
print("=" * 100)

print(summary)



KoBART 생성형 요약 모델 로드


loading file tokenizer.json from cache at C:\Users\AI_SVR05\.cache\huggingface\hub\models--digit82--kobart-summarization\snapshots\0887f7ac6e66df93248890f3460299d28bae1ddd\tokenizer.json
loading file tokenizer.model from cache at None
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at C:\Users\AI_SVR05\.cache\huggingface\hub\models--digit82--kobart-summarization\snapshots\0887f7ac6e66df93248890f3460299d28bae1ddd\special_tokens_map.json
loading file tokenizer_config.json from cache at C:\Users\AI_SVR05\.cache\huggingface\hub\models--digit82--kobart-summarization\snapshots\0887f7ac6e66df93248890f3460299d28bae1ddd\tokenizer_config.json
loading file chat_template.jinja from cache at None
loading configuration file config.json from cache at C:\Users\AI_SVR05\.cache\huggingface\hub\models--digit82--kobart-summarization\snapshots\0887f7ac6e66df93248890f3460299d28bae1ddd\config.json
You passed along `num_labels=3` with an incompatible id to lab

KoBART 로드 완료


Safetensors PR exists




추출된 핵심 문장

[1] 딥러닝은 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로,
대량의 데이터를 활용해 이미지 분류, 자연어 처리, 음성 인식 등의 다양한 문제를 해결할 수 있다.

[2] 특히 최근에는 생성형 인공지능 기술이 발전하면서 딥러닝의 중요성이 더욱 강조되고 있다.

[3] CNN 모델은 이미지의 공간적 특징을 효과적으로 추출할 수 있기 때문에
컴퓨터 비전 분야에서 매우 중요한 역할을 수행한다.

[4] 합성곱 연산을 활용하여 특징을 추출하며,
Pooling Layer를 통해 차원을 축소하고 연산량을 감소시킨다.

[5] Transformer 구조는 Self-Attention 메커니즘을 기반으로 문장 내 단어 간의 관계를 계산하며,
기존 RNN 기반 모델보다

[6] 특히 BERT와 GPT 같은 최신 자연어 처리 모델의 기반 구조로 사용된다.

[7] 데이터 전처리 과정에서는 결측치 제거, 정규화, 이상치 탐지 등의 작업이 수행된다.

[8] 이 과정은 모델의 성능을 향상시키기 위해 반드시 필요하다.

[9] GPU는 대규모 행렬 연산을 병렬로 처리할 수 있기 때문에
딥러닝 모델 학습 속도를 크게 향상시킬 수 있다.

[10] 특히 CUDA 환경 설정은 딥러닝 실습에서 매우 중요하게 다뤄진다.

[11] KoELECTRA는 ELECTRA 구조를 기반으로 학습된 한국어 언어 모델이며,
문장 분류와 감정 분석 분야에서 높은 성능을 보인다.


최종 생성형 요약
딥러닝은 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로, 인공지능 기술이 발전하면서 생성형 인공지능 기술의 중요성이 더욱 강조되고 있는 가운데, 최근에는 생성형 AI 기술이 발전함에 따라 빅데이터 전처리 과정에서는 결측치 제거, 정규화, 이상치 탐지 등의 작업이 수행된다.


요약 완료


최종 생성형 요약문
딥러닝은 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로, 인공지능 기술이 발전하면서 생성형 인공지능 기술의 중요성이 더욱 강

In [1]:
!pip install pymupdf transformers sentencepiece konlpy -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import fitz
import re
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BartForConditionalGeneration,
    PreTrainedTokenizerFast
)

from konlpy.tag import Okt
from collections import Counter

c:\Users\AI_SVR05\Desktop\NLP\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pdf_path = "./pdf/Chapter+1.+자연어처리의+개요.pdf"

electra_path = "./results/checkpoint-1923"

kobart_path = "./koelectra_keysentence_..."

In [4]:
doc = fitz.open(pdf_path)

full_text = ""

for page_num in range(len(doc)):

    page = doc.load_page(page_num)

    text = page.get_text()

    print(f"\n===== PAGE {page_num+1} =====")
    print(text[:300])

    full_text += text + "\n"

print("\n텍스트 추출 완료")


===== PAGE 1 =====
자연어처리의 개요 Ph.D김정순| 원더솔루션즈AI연구개발부연구소장동의대


===== PAGE 2 =====
Presentation Index 
Presentation 
Index 자연어처리의발전과정
01 자연어처리의주요Task
02 텍스트임베딩기법의진화
BoW(Bag-of-Word)
TF-IDF
Word2Vec
03 딥러닝기반언어모델과최신모델
Rule-based
Statistical
Deep Learning
04 
Cross-lingual
Hybrid Embedding
MultimodelEmbedding 
CNN, LSTM
Transformer
BERT & GPT
LLM(LLaMA, GPT-3, GPT-4 등)


===== PAGE 3 =====
자연어처리의  
발전과정 
History of Programming Language 


===== PAGE 4 =====
자연어처리의 발전 과정 
01 자연어처리(NLP)란무엇인가?•자연어처리(NLP)는인간이일상적으로사용하는언어를컴퓨터가


===== PAGE 5 =====
자연어처리의 발전 과정 
01 규칙기반(Rule-based)통계기반(Statistical)딥러닝(Deep Learning)개요•사람이직접언어규칙·사전을컴


===== PAGE 6 =====
자연어처리의 발전 과정 
01 RNNLSTMSeq2Seq(NPS 2014)Attention(ICLR 2015)Transformer(NPS 2017)GPT-1GPT-3BERTChatGPTFla


===== PAGE 7 =====
자연어처리의  
주요 Task 
History of Programming Language 
NLP가 실제로 해결하는 현실세계의 문제들 


===== PAGE 8 =====
자연어처리의 발전 과정 
02 
Classification
Generation
Understanding분류


===== PAGE 9 =====
텍스트 전처리와 토큰화 
02 왜전처리(Pre-P

In [6]:
# =========================================================
# 페이지별 텍스트 출력
# =========================================================

doc = fitz.open(pdf_path)

for page_num in range(len(doc)):

    page = doc.load_page(page_num)

    text = page.get_text()

    print("\n")
    print("="*80)
    print(f" PAGE {page_num+1} ")
    print("="*80)

    print(text)



 PAGE 1 
자연어처리의 개요 Ph.D김정순| 원더솔루션즈AI연구개발부연구소장동의대



 PAGE 2 
Presentation Index 
Presentation 
Index 자연어처리의발전과정
01 자연어처리의주요Task
02 텍스트임베딩기법의진화
BoW(Bag-of-Word)
TF-IDF
Word2Vec
03 딥러닝기반언어모델과최신모델
Rule-based
Statistical
Deep Learning
04 
Cross-lingual
Hybrid Embedding
MultimodelEmbedding 
CNN, LSTM
Transformer
BERT & GPT
LLM(LLaMA, GPT-3, GPT-4 등)
RLHF(인간피드백기반강화학습)
Classification
Generation
Understanding



 PAGE 3 
자연어처리의  
발전과정 
History of Programming Language 



 PAGE 4 
자연어처리의 발전 과정 
01 자연어처리(NLP)란무엇인가?•자연어처리(NLP)는인간이일상적으로사용하는언어를컴퓨터가



 PAGE 5 
자연어처리의 발전 과정 
01 규칙기반(Rule-based)통계기반(Statistical)딥러닝(Deep Learning)개요•사람이직접언어규칙·사전을컴



 PAGE 6 
자연어처리의 발전 과정 
01 RNNLSTMSeq2Seq(NPS 2014)Attention(ICLR 2015)Transformer(NPS 2017)GPT-1GPT-3BERTChatGPTFla



 PAGE 7 
자연어처리의  
주요 Task 
History of Programming Language 
NLP가 실제로 해결하는 현실세계의 문제들 



 PAGE 8 
자연어처리의 발전 과정 
02 
Classification
Generation
Understanding분류



 PAGE 9 
텍스트 전처리와 토큰화 
02 왜전처리(Pre-Processing)가필요한가?
•컴퓨

In [7]:
# =========================================================
# 추출 텍스트 txt 저장
# =========================================================

with open(
    "./extracted_text.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(full_text)

print("텍스트 저장 완료")

텍스트 저장 완료


In [8]:
sentences = re.split(
    r'(?<=[.!?다])\s+',
    full_text
)

sentences = [
    s.strip()
    for s in sentences
    if len(s.strip()) > 10
]

print("총 문장 수 :", len(sentences))

총 문장 수 : 4


In [10]:
import os

print(os.listdir("./results/checkpoint-1923"))

['config.json', 'model.safetensors', 'optimizer.pt', 'rng_state.pth', 'scheduler.pt', 'trainer_state.json', 'training_args.bin']


In [13]:
import os

print(os.listdir("./"))

['a.ipynb', 'check_dataset.py', 'check_json.py', 'extracted_text.txt', 'images', 'important_sentences.txt', 'koelectra_keysentence_model', 'koelectra_project', 'make_dataset.py', 'outputs', 'parse_document.py', 'pdf', 'preprocess_text.py', 'results', 'split_dataset.py', 'train.ipy', 'train_koelectra.py', 'venv']


In [14]:
import os

print(os.listdir("./koelectra_keysentence_model"))

['config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.txt']


In [35]:
# =========================================================
# PDF → Qwen 강의노트 생성기 (GPU 최적화 최종판)
# =========================================================

import fitz
import re
import gc
import torch
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================================================
# GPU 설정
# =========================================================

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA를 사용할 수 없습니다.")

device = "cuda:0"

# =========================================================
# Qwen 로드
# =========================================================

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    trust_remote_code=True
).to(device)

qwen_model.eval()

print("Qwen 로드 완료")
print("GPU:", torch.cuda.get_device_name(0))
print("모델 위치:", next(qwen_model.parameters()).device)

# =========================================================
# PDF 경로
# =========================================================

pdf_path = r"./pdf/Chapter+1.+자연어처리의+개요.pdf"

# =========================================================
# 텍스트 정리
# =========================================================

def clean_text(text):

    text = text.replace("\x00", " ")

    text = re.sub(r"\s+", " ", text)

    remove_chars = [
        "•", "◦", "▪", "▶", "◆", "■",
        "☞", "✓", "✔", "○", "●"
    ]

    for ch in remove_chars:
        text = text.replace(ch, " ")

    return text.strip()

# =========================================================
# 결과 후처리
# =========================================================

def postprocess(result):

    replacements = {

        "ディープ": "딥",
        "モデル": "모델",
        "トークン": "토큰",

        "デコーダ": "디코더",
        "エンコーダ": "인코더",

        "解讀": "이해",
        "解釈": "이해",

        "分類": "분류",
        "生成": "생성",

        "應用": "응용",
        "応用": "응용",

        " simultaneously ": " 동시에 "
    }

    for old, new in replacements.items():
        result = result.replace(old, new)

    result = result.replace("�", "")

    result = result.replace("###", "")
    result = result.replace("##", "")
    result = result.replace("#", "")

    result = result.replace("**", "")
    result = result.replace("*", "")

    result = result.replace("[", "")
    result = result.replace("]", "")

    result = result.replace("★", "")

    result = re.sub(r"\n{3,}", "\n\n", result)
    result = re.sub(r" +", " ", result)

    return result.strip()
# =========================================================
# 노트 생성
# =========================================================

def generate_note(text):

    prompt = f"""
너는 자연어처리 전공 교수이다.

다음 강의 슬라이드 내용을 시험 대비용 강의노트로 정리하라.

규칙

1. 한국어만 사용할 것
2. OCR 오류를 자동 수정할 것
3. 깨진 한글을 자연스럽게 복원할 것
4. 슬라이드 내용을 기반으로 설명할 것
5. 슬라이드에 없는 새로운 개념을 추가하지 말 것
6. 입력에 없는 수치, 연도, 인물, 모델명을 생성하지 말 것
7. 강의 슬라이드 원문을 그대로 반복 출력하지 말 것
8. 요약 및 설명 위주로 작성할 것
9. 중복 설명을 제거할 것
10. 문장이 중간에 끊기지 않도록 작성할 것
11. 여러 페이지의 내용이 연결되면 하나의 개념으로 통합할 것
12. Markdown 사용 금지
13. 특수문자 사용 금지
14. 일본어, 중국어, 한자는 한국어로 수정할 것
15. 강의 내용 외 추측 금지
16. 내용을 빠뜨리지 말 것
17. 강의 슬라이드 제목도 설명할 것
18. 출력이 중간에 끊기지 않도록 할 것

출력 형식

주제

주요 내용

1. 항목명
- 설명
- 설명

2. 항목명
- 설명
- 설명

3. 항목명
- 설명
- 설명

시험 포인트

1. 중요 개념
2. 중요 개념
3. 중요 개념

강의 슬라이드

{text}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=3500
    )

    inputs.pop("token_type_ids", None)

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    generated_ids = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    result = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    result = postprocess(result)

    return result

# =========================================================
# PDF 열기
# =========================================================

doc = fitz.open(pdf_path)

all_notes = ""

# =========================================================
# 2페이지 단위 처리
# =========================================================

chunk_size = 2

for start_page in range(
    0,
    len(doc),
    chunk_size
):

    end_page = min(
        start_page + chunk_size,
        len(doc)
    )

    print(
        f"{start_page + 1} ~ {end_page} 페이지 처리중..."
    )

    merged_text = ""

    for page_idx in range(
        start_page,
        end_page
    ):

        page = doc.load_page(page_idx)

        text = page.get_text()

        text = clean_text(text)

        if len(text.strip()) < 20:
            continue

        merged_text += (
            f"\n\n페이지 {page_idx + 1}\n"
        )

        merged_text += text

    if len(
        merged_text.strip()
    ) < 50:

        continue

    merged_text = merged_text[:4000]

    note = generate_note(
        merged_text
    )
    all_notes += "\n\n"
    all_notes += "=" * 100
    all_notes += "\n"
    all_notes += (
        f"페이지 {start_page + 1}"
        f" ~ "
        f"{end_page}"
    )
    all_notes += "\n"
    all_notes += "=" * 100
    all_notes += "\n\n"
    all_notes += note

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# =========================================================
# 저장
# =========================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = (
    f"study_note_qwen_{timestamp}.txt"
)

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(all_notes)

print()
print("=" * 60)
print("강의노트 저장 완료")
print(output_file)
print("=" * 60)
print()

doc.close()

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.22s/it]


Qwen 로드 완료
GPU: NVIDIA GeForce RTX 3070
모델 위치: cuda:0
1 ~ 2 페이지 처리중...
3 ~ 4 페이지 처리중...
5 ~ 6 페이지 처리중...
7 ~ 8 페이지 처리중...
9 ~ 10 페이지 처리중...
11 ~ 12 페이지 처리중...
13 ~ 14 페이지 처리중...
15 ~ 16 페이지 처리중...
17 ~ 18 페이지 처리중...
19 ~ 20 페이지 처리중...
21 ~ 22 페이지 처리중...
23 ~ 23 페이지 처리중...

강의노트 저장 완료
study_note_qwen_20260603_175519.txt



In [ ]:
# =========================================================
# PDF → Qwen 강의노트 생성기 (2단계 검수 버전)
# =========================================================

import fitz
import re
import gc
import torch
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================================================
# GPU 설정
# =========================================================

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA를 사용할 수 없습니다.")

device = "cuda:0"

# =========================================================
# Qwen 로드
# =========================================================

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    trust_remote_code=True
).to(device)

qwen_model.eval()

print("Qwen 로드 완료")
print("GPU:", torch.cuda.get_device_name(0))
print("모델 위치:", next(qwen_model.parameters()).device)

# =========================================================
# PDF 경로
# =========================================================

pdf_path = r"./pdf/Chapter+1.+자연어처리의+개요.pdf"

# =========================================================
# 텍스트 정리
# =========================================================

def clean_text(text):

    text = text.replace("\x00", " ")

    text = re.sub(r"\s+", " ", text)

    remove_chars = [
        "•", "◦", "▪", "▶", "◆", "■",
        "☞", "✓", "✔", "○", "●"
    ]

    for ch in remove_chars:
        text = text.replace(ch, " ")

    return text.strip()
# =========================================================
# 결과 후처리
# =========================================================

def postprocess(result):

    replacements = {

        "ディープ": "딥",
        "モデル": "모델",
        "トークン": "토큰",

        "デコーダ": "디코더",
        "エンコーダ": "인코더",

        "解讀": "이해",
        "解釈": "이해",

        "分類": "분류",
        "生成": "생성",

        "應用": "응용",
        "応用": "응용"
    }

    for old, new in replacements.items():
        result = result.replace(old, new)

    result = result.replace("�", "")

    result = result.replace("###", "")
    result = result.replace("##", "")
    result = result.replace("#", "")

    result = result.replace("**", "")
    result = result.replace("*", "")

    result = result.replace("[", "")
    result = result.replace("]", "")

    result = result.replace("★", "")

    result = re.sub(r"\n{3,}", "\n\n", result)
    result = re.sub(r" +", " ", result)

    result = result.replace(
    "강의 노트를 작성하면서",
    ""
    )

    result = result.replace(
        "강의노트 종료",
        ""
    )

    result = result.replace(
        "마지막 문장입니다.",
        ""
    )

    result = result.replace(
        "슬라이드에 구체적 설명 없음",
        ""
    )

    result = re.sub(r"\n{3,}", "\n\n", result)
    result = re.sub(r" +", " ", result)

    return result.strip()

# =========================================================
# 1차 요약
# =========================================================

def generate_note(text):

    prompt = f"""

너는 자연어처리 전공 교수이다.

다음 PPT 내용을
학생이 시험공부할 수 있도록
교수님의 필기노트처럼 설명하라.
이 페이지 제목을 먼저 찾고
그 제목에 맞게 정리하라

중요 규칙

1. PPT에 있는 내용만 사용할 것
2. 외부 지식 추가 금지
3. 예시 생성 금지
4. 설명 확장 금지
5. 슬라이드 내용을 보기 좋게 정리할 것
6. 시험에 나올 키워드를 모두 포함할 것
7. 원본 내용이 부족하면 부족한 그대로 작성할 것

출력 형식

■ 텍스트 임베딩 기법의 진화

[핵심 개념]

- One-hot Encoding
- TF-IDF
- Word2Vec
- GloVe
- FastText
- ELMo
- BERT

[정리]

One-hot Encoding은 ...
TF-IDF는 ...
Word2Vec은 ...

[시험 포인트]

★ Word2Vec
★ FastText
★ BERT

{text}
"""
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=3500
    )

    inputs.pop("token_type_ids", None)

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    generated_ids = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    result = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return postprocess(result)

# =========================================================
# PDF 열기
# =========================================================

doc = fitz.open(pdf_path)

all_notes = ""

chunk_size = 1

for start_page in range(
    0,
    len(doc),
    chunk_size
):

    end_page = min(
        start_page + chunk_size,
        len(doc)
    )

    print(
        f"{start_page + 1} ~ {end_page} 페이지 처리중..."
    )

    merged_text = ""

    for page_idx in range(
        start_page,
        end_page
    ):

        page = doc.load_page(page_idx)

        text = clean_text(
            page.get_text()
        )

        if len(text.split()) < 20:
            continue

        merged_text += (
            f"\n\n페이지 {page_idx + 1}\n"
        )

        merged_text += text

    if len(merged_text.strip()) < 50:
        continue

    merged_text = merged_text[:3000]

    note = generate_note(
        merged_text
    )

    all_notes += "\n\n"
    all_notes += "=" * 100
    all_notes += "\n"
    all_notes += (
        f"페이지 {start_page + 1}"
        f" ~ "
        f"{end_page}"
    )
    all_notes += "\n"
    all_notes += "=" * 100
    all_notes += "\n\n"
    all_notes += note

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = (
    f"study_note_qwen_{timestamp}.txt"
)

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(all_notes)

print()
print("=" * 60)
print("강의노트 저장 완료")
print(output_file)
print("=" * 60)

doc.close()

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.05s/it]


Qwen 로드 완료
GPU: NVIDIA GeForce RTX 3070
모델 위치: cuda:0
1 ~ 4 페이지 처리중...
5 ~ 8 페이지 처리중...
9 ~ 12 페이지 처리중...
13 ~ 16 페이지 처리중...
17 ~ 20 페이지 처리중...
21 ~ 23 페이지 처리중...

강의노트 저장 완료
study_note_qwen_20260604_143303.txt
